In [ ]:
!wget https://archive.ics.uci.edu/static/public/222/bank+marketing.zip -O bank_marketing.zip
!unzip -o bank_marketing.zip -d bank_marketing
!unzip -o bank_marketing/bank-additional.zip -d bank_marketing

--2026-09-23 17:08:12--  https://archive.ics.uci.edu/static/public/222/bank+marketing.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘bank_marketing.zip’

bank_marketing.zip      [  <=>               ] 999.85K  2.70MB/s    in 0.4s    

2026-09-23 17:08:14 (2.70 MB/s) - ‘bank_marketing.zip’ saved [1023843]

Archive:  bank_marketing.zip
 extracting: bank_marketing/bank.zip  
 extracting: bank_marketing/bank-additional.zip  
Archive:  bank_marketing/bank-additional.zip
   creating: bank_marketing/bank-additional/
  inflating: bank_marketing/bank-additional/.DS_Store  
   creating: bank_marketing/__MACOSX/
   creating: bank_marketing/__MACOSX/bank-additional/
  inflating: bank_marketing/__MACOSX/bank-additional/._.DS_Store  
  inflating: bank_marketing/bank-additional/.Rhistory  
  inflating: bank_

In [ ]:
import pandas as pd

df = pd.read_csv(
    "bank_marketing/bank-additional/bank-additional-full.csv",
    sep=";"
)

In [ ]:
print(df.shape)
print(df.columns.tolist())
print(df['y'].value_counts())
print(df.isnull().sum())

(41188, 21)
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']
y
no     36548
yes     4640
Name: count, dtype: int64
age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64


In [ ]:
# Standardize column names
df.columns = df.columns.str.strip()

# Convert target to binary
df['Converted'] = df['y'].map({'yes': 1, 'no': 0})

# Flag "unknown" as its own explicit category rather than silently dropping rows
# (dropping would lose ~20%+ of data — better to keep and treat Unknown as a segment)
for col in ['job', 'education', 'default', 'housing', 'loan']:
    df[col] = df[col].replace('unknown', 'Unknown')

# Drop duplicates
df = df.drop_duplicates()

In [ ]:
# Stage 1: Contacted — every row qualifies (baseline)
df['Stage_Contacted'] = 1

# Stage 2: Engaged — previously contacted OR contacted more than once this campaign
df['Stage_Engaged'] = ((df['pdays'] != 999) | (df['campaign'] > 1)).astype(int)
# Note: in bank-additional-full, 999 means "not previously contacted" (not -1 like the older version)

# Stage 3: Converted — subscribed to term deposit
df['Stage_Converted'] = df['Converted']

# Funnel-friendly categorical stage label per row (for cohort-style breakdowns)
def funnel_stage(row):
    if row['Stage_Converted'] == 1:
        return 'Converted'
    elif row['Stage_Engaged'] == 1:
        return 'Engaged (Not Converted)'
    else:
        return 'Contacted Only'

df['Funnel Stage'] = df.apply(funnel_stage, axis=1)

In [ ]:
funnel_summary = pd.DataFrame({
    'Stage': ['Contacted', 'Engaged', 'Converted'],
    'Count': [
        df['Stage_Contacted'].sum(),
        df['Stage_Engaged'].sum(),
        df['Stage_Converted'].sum()
    ]
})
funnel_summary['Conversion Rate %'] = (
    funnel_summary['Count'] / funnel_summary['Count'].iloc[0] * 100
)
funnel_summary['Drop-off %'] = 100 - funnel_summary['Conversion Rate %']
funnel_summary.to_csv('funnel_summary.csv', index=False)

In [ ]:
channel_summary = df.groupby('contact').agg(
    Total_Contacted=('Stage_Contacted', 'sum'),
    Total_Converted=('Stage_Converted', 'sum')
).reset_index()
channel_summary['Conversion Rate %'] = (
    channel_summary['Total_Converted'] / channel_summary['Total_Contacted'] * 100
)
channel_summary.to_csv('channel_summary.csv', index=False)

In [ ]:
df.to_csv('cleaned_bank_marketing.csv', index=False)

from google.colab import files
files.download('cleaned_bank_marketing.csv')
files.download('funnel_summary.csv')
files.download('channel_summary.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>